# PRIMA NER — Run 2

**4 systèmes** — un par cellule de code :

| # | Modèle | Paramètres | epochs | batch | LR / max_len |
|---|--------|-----------|--------|-------|--------------|
| 1 | XLM-RoBERTa | **nos paramètres** | 20 | 8 | 2e-5 / 128 |
| 2 | XLM-RoBERTa | **article** (Torres Aguilar 2022) | 5 | 16 | 2e-5 / 250 |
| 3 | BiLSTM-CRF | **nos paramètres** | 50 | 8 | 0.001 |
| 4 | BiLSTM-CRF | **article** | 5 | 16 | 0.01 |

⚠️ Exécuter **Cell 0** en premier (montage Drive + pip install).

Sorties → `Mon Drive/Prima/run2/<tag>/logs/` et `.../model/`

In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 0 — Montage Google Drive + installation des dépendances
# À exécuter en PREMIER dans chaque nouvelle session Colab.
# ═══════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

!pip install -q transformers seqeval
print('Dépendances installées.')


Mounted at /content/drive
GPU disponible : True
GPU : Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Dépendances installées.


## Cell 1 — XLM-RoBERTa (nos paramètres)
`epochs=20, batch=8, max_len=128`  →  `Prima/run2/xlmroberta/`

In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL — XLM-RoBERTa — NOS paramètres (epochs=20, batch=8, max_len=128)
# Données  : /content/drive/MyDrive/Prima/ner_data/
# Sorties  : /content/drive/MyDrive/Prima/run2/xlmroberta/
# ═══════════════════════════════════════════════════════════════

# -*- coding: utf-8 -*-
"""
prima_xlmroberta.py
===================
Fine-tuning XLM-RoBERTa pour la NER sur les entrées du catalogue Riccardiana.

Labels : auteur, ouvrage, material, date, etat  (schéma BIO)
Données : train.json / val.json / test.json  (gold_annotations_with_el.json)

Sorties dans ./logs/ et ./model_xlmroberta/ :
  - logs/training_log.jsonl     → loss + F1 + temps par epoch
  - logs/eval_results.json      → métriques finales sur val
  - logs/test_results.json      → métriques finales sur test
  - model_xlmroberta/           → modèle fine-tuné (sauvegardé)

Usage (dans ~/prima_ner/xlm_roberta/) :
    python3 prima_xlmroberta.py

Dépendances :
    pip install transformers datasets seqeval torch

================================================================================
【代码逻辑与结构说明】
================================================================================

本脚本针对 PRIMA 项目（ERC 101142242）对里恰尔迪亚纳图书馆中世纪手稿目录
进行命名实体识别（NER）的微调训练。

■ 任务目标
  识别拉丁语/意大利语目录条目中的5类实体：
    - auteur   : 作者名（拉丁属格或意大利语形式）
    - ouvrage  : 作品名
    - material : 载体描述（如 Cod. membr. in fol.）
    - date     : 手稿年代（如 Saec. XIV）
    - etat     : 保存状态描述

■ 数据
  - 来源：gold_annotations_with_el.json（100条人工标注）
  - 划分：train(70) / val(20) / test(10)，seed=42，随机划分
  - 标注格式：每条 entry 含 raw 文本 + spans（含 text、label 字段）

■ 模型
  XLM-RoBERTa-base（多语言预训练模型，支持拉丁语/法语/意大利语）
  + TokenClassification head（BIO标签分类层）

■ 代码结构（按执行顺序）
  1. json_to_bio()     : 将 JSON 标注转换为 BIO 序列
                         - 按字符位置构建 char_labels 数组
                         - 按空格切词，取每个词首字符的标签
  2. NERDataset        : PyTorch Dataset
                         - 使用 tokenizer 将词列表转为 subword token
                         - 对齐 BIO 标签（subword → word_ids 映射）
                         - padding 到 MAX_LEN=128，-100 掩盖非实体位置
  3. bio_to_spans()    : 将 BIO 序列还原为实体 span 列表 {text, label}
  4. evaluate()        : 使用 seqeval 计算实体级别的 P/R/F1
                         - 按类型单独报告（auteur/ouvrage/material/date/etat）
                         - 同时计算 macro 平均
                         - 可选 entries 参数：记录每条 entry 的 gold/pred span 明细
  5. 数据加载          : 读取 train/val/test.json，转 BIO，构建 DataLoader
  6. 模型初始化        : 加载 xlm-roberta-base，替换分类头（NUM_LABELS=11）
  7. 优化器            : AdamW，lr=2e-5，线性 warmup（前10%步骤）
  8. 训练循环          : 每个 epoch：
                         - 前向传播 + 计算 cross-entropy loss
                         - 梯度裁剪（max_norm=1.0）
                         - 在 val 上评估 F1（含 entry 明细）
                         - 保存最佳模型（按 val F1）
                         - 记录 epoch 日志（loss/F1/时间戳）到 JSONL 文件
  9. 测试评估          : 加载最佳模型，同时在 val 和 test 上计算最终指标
                         - 输出 test_results.json（含 val+test 按类型报告和 entry 明细）
  10. make_markdown_report() : 生成按错误类型分组的 Markdown 报告
                         - correct_exact / correct_vide / faux_negatif /
                           faux_positif / erreur_partielle
                         - val 和 test 各有独立目录（resultat_val / resultat_test）

■ 超参数
  EPOCHS=20, BATCH_SIZE=8, LR=2e-5, MAX_LEN=128, SEED=42

■ 输出文件
  logs/training_log.jsonl          → 每个 epoch 的 loss、val F1、时间戳
  logs/test_results.json           → val+test 完整结果（含 entry 明细）
  logs/analysis_report.md          → val+test 合并 Markdown 报告
  logs/resultat_val/               → val 独立目录（JSON + Markdown）
  logs/resultat_test/              → test 独立目录（JSON + Markdown）
  model_xlmroberta/                → 最佳模型权重（可用于推理或继续训练）
================================================================================
"""

import json
import os
import time
from datetime import datetime
from collections import defaultdict

import numpy as np


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME   = "xlm-roberta-base"   # 多语言预训练模型（支持100+语言含拉丁语）
DATA_DIR  = "/content/drive/MyDrive/Prima/ner_data"
LOG_DIR   = "/content/drive/MyDrive/Prima/run2/xlmroberta/logs"
MODEL_DIR = "/content/drive/MyDrive/Prima/run2/xlmroberta/model"

EPOCHS       = 20       # nos paramètres
BATCH_SIZE   = 8        # nos paramètres
LR           = 2e-5     # 学习率（Transformer fine-tuning 标准值）
MAX_LEN      = 128      # nos paramètres
SEED         = 42       # 随机种子（与数据划分保持一致）

# BIO 标签体系：O + 每类实体的 B-/I- 标签
# O=0, B-auteur=1, I-auteur=2, B-ouvrage=3, I-ouvrage=4, ...
LABELS = ["auteur", "ouvrage", "material", "date", "etat"]
LABEL2ID = {"O": 0}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID)
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)  # = 11

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
VAL_DIR  = os.path.join(LOG_DIR, "resultat_val")
TEST_DIR = os.path.join(LOG_DIR, "resultat_test")
os.makedirs(VAL_DIR,  exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Device: {device}")
print(f"[Config] Model : {MODEL_NAME}")
print(f"[Config] Epochs: {EPOCHS}, LR: {LR}, Batch: {BATCH_SIZE}")


# ── 1. Conversion JSON → BIO ──────────────────────────────────────────────────
def json_to_bio(entries):
    """
    将 JSON 标注转换为 BIO 序列。
    输入：entries 列表，每条含 raw（原文）和 spans（标注片段列表）
    输出：[(tokens, labels), ...] 列表
      - tokens : 按空格切分的词列表
      - labels : 对应的 BIO 标签列表（如 B-auteur, I-auteur, O）
    """
    samples = []
    for entry in entries:
        raw = entry["raw"]
        spans = entry.get("spans", [])

        # 步骤1：按字符构建标签数组，初始全为 "O"
        # 每个字符位置对应一个标签，span 首字符标 B-，其余字符标 I-
        char_labels = ["O"] * len(raw)
        for span in spans:
            text  = span["text"]
            label = span["label"]
            idx   = raw.find(text)
            if idx == -1:
                continue  # 找不到则跳过（理论上不会发生）
            char_labels[idx] = f"B-{label}"           # 实体首字符
            for i in range(idx + 1, idx + len(text)):
                char_labels[i] = f"I-{label}"         # 实体内部字符

        # 步骤2：按空格切词，取每个词首字符的标签作为该词的标签
        # 这是"词级别"的 BIO 标注，与 subword tokenizer 的对齐在 NERDataset 中处理
        tokens = []
        labels = []
        pos = 0
        for word in raw.split():
            idx = raw.find(word, pos)
            if idx == -1:
                tokens.append(word)
                labels.append("O")
                pos += len(word)
                continue
            tok_label = char_labels[idx]  # 取词首字符的标签
            tokens.append(word)
            labels.append(tok_label)
            pos = idx + len(word)

        samples.append((tokens, labels))
    return samples


# ── 2. Dataset ────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    """
    PyTorch Dataset，将词级别的 BIO 序列转换为模型输入。

    核心问题：XLM-RoBERTa 使用 SentencePiece 切分 subword，
    一个词可能被切成多个 token（如 "Augustini" → "Aug", "ust", "ini"）。
    需要将词级别的 BIO 标签对齐到 subword 级别：
      - 词的第一个 subword 保留原标签
      - 后续 subword：若原标签是 B-X 则改为 I-X，否则保持不变
      - 特殊 token（[CLS], [SEP], [PAD]）标为 -100（loss 计算时忽略）
    """
    def __init__(self, samples, tokenizer, max_len):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens, labels = self.samples[idx]

        # subword tokenization，is_split_into_words=True 告知 tokenizer 输入已切词
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        # word_ids()：每个 subword token 对应原来第几个词（None 表示特殊 token）
        word_ids    = encoding.word_ids()
        label_ids   = []
        prev_word   = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)          # 特殊 token，loss 中忽略
            elif wid != prev_word:
                label_ids.append(LABEL2ID[labels[wid]])   # 词的第一个 subword
            else:
                # 同一个词的后续 subword：B- 改为 I-，其余不变
                l = labels[wid]
                if l.startswith("B-"):
                    l = "I-" + l[2:]
                label_ids.append(LABEL2ID.get(l, 0))
            prev_word = wid

        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(label_ids, dtype=torch.long),
        }


# ── 3. Évaluation seqeval ────────────────────────────────────────────────────
def bio_to_spans(tokens, bio):
    """
    将 BIO 序列还原为实体 span 列表。
    输入：tokens（词列表）, bio（对应的 BIO 标签列表）
    输出：[{"text": "...", "label": "..."}, ...]
    """
    spans = []
    cur_text, cur_label = [], None
    for tok, tag in zip(tokens, bio):
        if tag.startswith("B-"):
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text  = [tok]
            cur_label = tag[2:]
        elif tag.startswith("I-") and cur_text:
            cur_text.append(tok)
        else:
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text, cur_label = [], None
    if cur_text:
        spans.append({"text": " ".join(cur_text), "label": cur_label})
    return spans


def evaluate(model, loader, split_name="val", entries=None):
    """
    使用 seqeval 在实体级别（而非 token 级别）计算 P/R/F1。
    seqeval 会将连续的 B-X I-X 序列识别为一个完整实体再比较，
    比 token 级别的准确率更符合 NER 任务的实际需求。

    参数：
      entries : 若传入原始 JSON 条目列表，则记录每条 entry 的
                gold_spans 和 pred_spans 明细（用于 Markdown 报告）

    输出：
      - 终端打印每个类型的 precision/recall/F1
      - 返回 dict：{f1, precision, recall, report, details}
    """
    model.eval()
    all_preds = []
    all_trues = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            lab       = batch["labels"].to(device)
            outputs   = model(input_ids=input_ids, attention_mask=attn_mask)
            logits    = outputs.logits
            preds     = torch.argmax(logits, dim=-1)   # 取概率最高的标签

            for pred_seq, true_seq in zip(preds.cpu().tolist(), lab.cpu().tolist()):
                p_tags = []
                t_tags = []
                for p, t in zip(pred_seq, true_seq):
                    if t == -100:
                        continue   # 跳过特殊 token
                    p_tags.append(ID2LABEL[p])
                    t_tags.append(ID2LABEL[t])
                all_preds.append(p_tags)
                all_trues.append(t_tags)

    report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
    f1  = f1_score(all_trues, all_preds, zero_division=0)
    pre = precision_score(all_trues, all_preds, zero_division=0)
    rec = recall_score(all_trues, all_preds, zero_division=0)
    print(f"  [{split_name}] F1={f1:.4f}  P={pre:.4f}  R={rec:.4f}")
    print(classification_report(all_trues, all_preds, zero_division=0))

    # entry 级别明细（用于 Markdown 报告的错误分析）
    details = []
    if entries is not None:
        for entry, gold_bio, pred_bio in zip(entries, all_trues, all_preds):
            tokens = entry["raw"].split()
            details.append({
                "entry_id":   entry["entry_id"],
                "raw":        entry["raw"],
                "gold_spans": bio_to_spans(tokens, gold_bio),
                "pred_spans": bio_to_spans(tokens, pred_bio),
            })

    return {"f1": f1, "precision": pre, "recall": rec,
            "report": report, "details": details}


# ── 4. Chargement des données ─────────────────────────────────────────────────
print("\n[1/5] Chargement des données...")
for split in ["train", "val", "test"]:
    path = os.path.join(DATA_DIR, f"{split}.json")
    with open(path, encoding="utf-8") as f:
        globals()[f"{split}_entries"] = json.load(f)
    print(f"  {split}: {len(globals()[f'{split}_entries'])} entrées")

train_samples = json_to_bio(train_entries)
val_samples   = json_to_bio(val_entries)
test_samples  = json_to_bio(test_entries)
print(f"  BIO → train:{len(train_samples)}, val:{len(val_samples)}, test:{len(test_samples)}")


# ── 5. Tokenizer & Dataset ────────────────────────────────────────────────────
print("\n[2/5] Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = NERDataset(train_samples, tokenizer, MAX_LEN)
val_dataset   = NERDataset(val_samples,   tokenizer, MAX_LEN)
test_dataset  = NERDataset(test_samples,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)


# ── 6. Modèle ─────────────────────────────────────────────────────────────────
print("\n[3/5] Chargement du modèle...")
# 加载 xlm-roberta-base 并替换分类头：
# 原始模型输出 768 维向量，分类头将其映射到 NUM_LABELS=11 个标签
# （O + 5类×2 = 11：O, B-auteur, I-auteur, B-ouvrage, ... B-etat, I-etat）
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

# AdamW：Adam + weight decay，适合 Transformer fine-tuning
optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
# 线性 warmup：前10%的步骤学习率从0线性增加到LR，之后线性衰减到0
# 避免训练初期大学习率破坏预训练权重
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)


# ── 7. Entraînement ───────────────────────────────────────────────────────────
print(f"\n[4/5] Entraînement ({EPOCHS} epochs)...")
log_path = os.path.join(LOG_DIR, "training_log.jsonl")
best_f1  = 0.0
training_start = time.time()

with open(log_path, "w", encoding="utf-8") as logf:
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels    = batch["labels"].to(device)

            # 前向传播：模型自动计算 cross-entropy loss（-100 位置被忽略）
            outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            loss    = outputs.loss
            total_loss += loss.item()

            # 反向传播
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 梯度裁剪防止梯度爆炸
            optimizer.step()
            scheduler.step()   # 线性 warmup + 衰减学习率
            optimizer.zero_grad()

        avg_loss   = total_loss / len(train_loader)
        epoch_time = time.time() - epoch_start
        timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        print(f"\nEpoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  time={epoch_time:.1f}s  [{timestamp}]")
        val_metrics = evaluate(model, val_loader, "val", entries=val_entries)

        # 保存 val F1 最高的模型（而非最后一个 epoch）
        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            model.save_pretrained(MODEL_DIR)
            tokenizer.save_pretrained(MODEL_DIR)
            print(f"  ✓ Meilleur modèle sauvegardé (F1={best_f1:.4f})")

        # 每个 epoch 写一行 JSON 日志（含时间戳，便于追踪训练进度）
        log_entry = {
            "epoch":         epoch,
            "timestamp":     timestamp,          # 该 epoch 结束的时刻
            "duration_s":    round(epoch_time, 2),
            "train_loss":    round(avg_loss, 6),
            "val_f1":        round(val_metrics["f1"], 6),
            "val_precision": round(val_metrics["precision"], 6),
            "val_recall":    round(val_metrics["recall"], 6),
        }
        logf.write(json.dumps(log_entry, ensure_ascii=False) + "\n")
        logf.flush()   # 实时写入，不等训练结束

total_time = time.time() - training_start
print(f"\nEntraînement terminé en {total_time/60:.1f} min | Meilleur val F1 = {best_f1:.4f}")


# ── 8. Évaluation finale sur val + test ──────────────────────────────────────
print("\n[5/5] Évaluation sur val et test sets...")
# Recharger le meilleur modèle
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
model.to(device)
val_metrics  = evaluate(model, val_loader,  "val",  entries=val_entries)
test_metrics = evaluate(model, test_loader, "test", entries=test_entries)

# Sauvegarder les résultats
results = {
    "model":       MODEL_NAME,
    "epochs":      EPOCHS,
    "lr":          LR,
    "batch_size":  BATCH_SIZE,
    "seed":        SEED,
    "best_val_f1": round(best_f1, 6),
    "val_metrics": {
        "f1":        round(val_metrics["f1"], 6),
        "precision": round(val_metrics["precision"], 6),
        "recall":    round(val_metrics["recall"], 6),
    },
    "val_report_by_label":  val_metrics["report"],
    "val_details":          val_metrics["details"],
    "test_metrics": {
        "f1":        round(test_metrics["f1"], 6),
        "precision": round(test_metrics["precision"], 6),
        "recall":    round(test_metrics["recall"], 6),
    },
    "test_report_by_label": test_metrics["report"],
    "test_details":         test_metrics["details"],
    "total_training_time_min": round(total_time / 60, 2),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

with open(os.path.join(LOG_DIR, "test_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)

# ── Rapport Markdown ──────────────────────────────────────────────────────────
LABELS_ORDER = ["auteur", "ouvrage", "material", "date", "etat"]


def error_type(gold_spans, pred_spans):
    gold_set = {(s["text"], s["label"]) for s in gold_spans}
    pred_set = {(s["text"], s["label"]) for s in pred_spans}
    if not gold_set and not pred_set:
        return "correct_vide"
    if gold_set == pred_set:
        return "correct_exact"
    if gold_set and not pred_set:
        return "faux_negatif"
    if pred_set and not gold_set:
        return "faux_positif"
    return "erreur_partielle"


def make_markdown_report(split_name, metrics, details, report, out_dir=None):
    lines = []
    lines.append(f"# Rapport d'analyse — XLM-RoBERTa ({split_name.upper()})\n")
    lines.append(f"**Paramètres** : epochs={EPOCHS}, lr={LR}, batch={BATCH_SIZE}, max_len={MAX_LEN}  ")
    lines.append(f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    lines.append("---\n")
    lines.append("## Matrice de résultats (seqeval — entité stricte)\n")
    lines.append("```")
    lines.append(f"{'Label':<15} {'P':>8} {'R':>8} {'F1':>8} {'Support':>9}")
    lines.append("-" * 52)
    for lab in LABELS_ORDER:
        r = report.get(lab, {})
        lines.append(f"{lab:<15} {r.get('precision',0):>8.3f} {r.get('recall',0):>8.3f} "
                     f"{r.get('f1-score',0):>8.3f} {int(r.get('support',0)):>9}")
    lines.append("-" * 52)
    for avg in ["micro avg", "macro avg", "weighted avg"]:
        r = report.get(avg, {})
        lines.append(f"{avg:<15} {r.get('precision',0):>8.3f} {r.get('recall',0):>8.3f} "
                     f"{r.get('f1-score',0):>8.3f} {int(r.get('support',0)):>9}")
    lines.append("```\n")
    lines.append(f"- **F1** = {metrics['f1']:.4f}  **P** = {metrics['precision']:.4f}  **R** = {metrics['recall']:.4f}\n")
    lines.append("---\n")
    lines.append("## Exemples détaillés par type d'erreur\n")
    grouped = {"correct_exact": [], "correct_vide": [],
               "faux_negatif": [], "faux_positif": [], "erreur_partielle": []}
    for d in details:
        t = error_type(d["gold_spans"], d["pred_spans"])
        grouped[t].append(d)
    labels_fr = {
        "correct_exact":    "✅ Correct — correspondance exacte",
        "correct_vide":     "✅ Correct — aucune entité attendue ni prédite",
        "faux_negatif":     "❌ Faux négatif — entité non détectée",
        "faux_positif":     "⚠️ Faux positif — entité prédite sans gold",
        "erreur_partielle": "〰️ Erreur partielle — détection incomplète ou mauvais label",
    }
    for key, title in labels_fr.items():
        entries_g = grouped[key]
        if not entries_g:
            continue
        lines.append(f"### {title} ({len(entries_g)} entrées)\n")
        for d in entries_g:
            gold_str = ", ".join(f"{s['label']}({s['text']})" for s in d["gold_spans"]) or "—"
            pred_str = ", ".join(f"{s['label']}({s['text']})" for s in d["pred_spans"]) or "—"
            lines.append(f"**#{d['entry_id']}** `{d['raw'][:90]}`")
            lines.append(f"- Gold : {gold_str}")
            lines.append(f"- Préd : {pred_str}\n")
    report_content = "\n".join(lines) + "\n"
    if out_dir:
        md_path = os.path.join(out_dir, f"analysis_{split_name}.md")
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        print(f"  → Rapport {split_name} : {md_path}")
        json_path = os.path.join(out_dir, f"results_{split_name}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({"split": split_name, "metrics": metrics,
                       "report_by_label": report, "details": details},
                      f, ensure_ascii=False, indent=2, cls=NumpyEncoder)
        print(f"  → JSON {split_name}    : {json_path}")
    return report_content


make_markdown_report("val",  val_metrics,  val_metrics["details"],  val_metrics["report"],  VAL_DIR)
make_markdown_report("test", test_metrics, test_metrics["details"], test_metrics["report"], TEST_DIR)

# Rapport global (val + test ensemble)
md_lines = []
md_lines.append("# Rapport d'analyse — XLM-RoBERTa (nos paramètres)\n")
md_lines.append(f"**Paramètres** : epochs={EPOCHS}, lr={LR}, batch={BATCH_SIZE}, max_len={MAX_LEN}  ")
md_lines.append(f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
md_lines.append("---\n")
md_lines.append(make_markdown_report("val",  val_metrics,  val_metrics["details"],  val_metrics["report"]))
md_lines.append("---\n")
md_lines.append(make_markdown_report("test", test_metrics, test_metrics["details"], test_metrics["report"]))

md_path = os.path.join(LOG_DIR, "analysis_report.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines) + "\n")

print(f"\n[Done] Val  F1={val_metrics['f1']:.4f}")
print(f"       Test F1={test_metrics['f1']:.4f}")
print(f"       Rapport global → {md_path}")
print(f"       Val            → {VAL_DIR}/")
print(f"       Test           → {TEST_DIR}/")
print(f"       Modèle         → {MODEL_DIR}/")


[Config] Device: cuda
[Config] Model : xlm-roberta-base
[Config] Epochs: 20, LR: 2e-05, Batch: 8

[1/5] Chargement des données...
  train: 70 entrées
  val: 20 entrées
  test: 10 entrées
  BIO → train:70, val:20, test:10

[2/5] Chargement du tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

  ✓ Meilleur modèle sauvegardé (F1=0.8068)

Epoch 17/20  loss=0.0962  time=2.4s  [2026-06-07 22:33:50]
  [val] F1=0.8023  P=0.7889  R=0.8161
              precision    recall  f1-score   support

      auteur       0.75      0.71      0.73        17
        date       1.00      1.00      1.00        20
        etat       0.67      0.67      0.67         6
    material       0.95      0.95      0.95        21
     ouvrage       0.56      0.65      0.60        23

   micro avg       0.79      0.82      0.80        87
   macro avg       0.78      0.80      0.79        87
weighted avg       0.80      0.82      0.81        87


Epoch 18/20  loss=0.0879  time=2.4s  [2026-06-07 22:33:52]
  [val] F1=0.7826  P=0.7423  R=0.8276
              precision    recall  f1-score   support

      auteur       0.72      0.76      0.74        17
        date       1.00      1.00      1.00        20
        etat       0.67      0.67      0.67         6
    material       0.95      0.95      0.95        21
 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Meilleur modèle sauvegardé (F1=0.8156)

Epoch 20/20  loss=0.0769  time=2.4s  [2026-06-07 22:34:36]
  [val] F1=0.8156  P=0.7935  R=0.8391
              precision    recall  f1-score   support

      auteur       0.81      0.76      0.79        17
        date       1.00      1.00      1.00        20
        etat       0.67      0.67      0.67         6
    material       0.95      0.95      0.95        21
     ouvrage       0.55      0.70      0.62        23

   micro avg       0.79      0.84      0.82        87
   macro avg       0.80      0.82      0.80        87
weighted avg       0.81      0.84      0.82        87


Entraînement terminé en 5.0 min | Meilleur val F1 = 0.8156

[5/5] Évaluation sur val et test sets...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [val] F1=0.8156  P=0.7935  R=0.8391
              precision    recall  f1-score   support

      auteur       0.81      0.76      0.79        17
        date       1.00      1.00      1.00        20
        etat       0.67      0.67      0.67         6
    material       0.95      0.95      0.95        21
     ouvrage       0.55      0.70      0.62        23

   micro avg       0.79      0.84      0.82        87
   macro avg       0.80      0.82      0.80        87
weighted avg       0.81      0.84      0.82        87

  [test] F1=0.7200  P=0.6667  R=0.7826
              precision    recall  f1-score   support

      auteur       0.53      0.80      0.64        10
        date       1.00      1.00      1.00        10
        etat       0.50      0.50      0.50         4
    material       0.92      1.00      0.96        11
     ouvrage       0.38      0.45      0.42        11

   micro avg       0.67      0.78      0.72        46
   macro avg       0.67      0.75      0.70        46


## Cell 2 — XLM-RoBERTa (paramètres article)
`epochs=5, batch=16, max_len=250`  →  `Prima/run2/xlmroberta_article/`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL — XLM-RoBERTa — paramètres ARTICLE (epochs=5, batch=16, max_len=250)
# Données  : /content/drive/MyDrive/Prima/ner_data/
# Sorties  : /content/drive/MyDrive/Prima/run2/xlmroberta_article/
# ═══════════════════════════════════════════════════════════════

# -*- coding: utf-8 -*-
"""
prima_xlmroberta.py
===================
Fine-tuning XLM-RoBERTa pour la NER sur les entrées du catalogue Riccardiana.

Labels : auteur, ouvrage, material, date, etat  (schéma BIO)
Données : train.json / val.json / test.json  (gold_annotations_with_el.json)

Sorties dans ./logs/ et ./model_xlmroberta/ :
  - logs/training_log.jsonl     → loss + F1 + temps par epoch
  - logs/eval_results.json      → métriques finales sur val
  - logs/test_results.json      → métriques finales sur test
  - model_xlmroberta/           → modèle fine-tuné (sauvegardé)

Usage (dans ~/prima_ner/xlm_roberta/) :
    python3 prima_xlmroberta.py

Dépendances :
    pip install transformers datasets seqeval torch

================================================================================
【代码逻辑与结构说明】
================================================================================

本脚本针对 PRIMA 项目（ERC 101142242）对里恰尔迪亚纳图书馆中世纪手稿目录
进行命名实体识别（NER）的微调训练。

■ 任务目标
  识别拉丁语/意大利语目录条目中的5类实体：
    - auteur   : 作者名（拉丁属格或意大利语形式）
    - ouvrage  : 作品名
    - material : 载体描述（如 Cod. membr. in fol.）
    - date     : 手稿年代（如 Saec. XIV）
    - etat     : 保存状态描述

■ 数据
  - 来源：gold_annotations_with_el.json（100条人工标注）
  - 划分：train(70) / val(20) / test(10)，seed=42，随机划分
  - 标注格式：每条 entry 含 raw 文本 + spans（含 text、label 字段）

■ 模型
  XLM-RoBERTa-base（多语言预训练模型，支持拉丁语/法语/意大利语）
  + TokenClassification head（BIO标签分类层）

■ 代码结构（按执行顺序）
  1. json_to_bio()     : 将 JSON 标注转换为 BIO 序列
                         - 按字符位置构建 char_labels 数组
                         - 按空格切词，取每个词首字符的标签
  2. NERDataset        : PyTorch Dataset
                         - 使用 tokenizer 将词列表转为 subword token
                         - 对齐 BIO 标签（subword → word_ids 映射）
                         - padding 到 MAX_LEN=128，-100 掩盖非实体位置
  3. evaluate()        : 使用 seqeval 计算实体级别的 P/R/F1
                         - 按类型单独报告（auteur/ouvrage/material/date/etat）
                         - 同时计算 macro 平均
  4. 数据加载          : 读取 train/val/test.json，转 BIO，构建 DataLoader
  5. 模型初始化        : 加载 xlm-roberta-base，替换分类头（NUM_LABELS=11）
  6. 优化器            : AdamW，lr=2e-5，线性 warmup（前10%步骤）
  7. 训练循环          : 每个 epoch：
                         - 前向传播 + 计算 cross-entropy loss
                         - 梯度裁剪（max_norm=1.0）
                         - 在 val 上评估 F1
                         - 保存最佳模型（按 val F1）
                         - 记录 epoch 日志（loss/F1/时间戳）到 JSONL 文件
  8. 测试评估          : 加载最佳模型，在 test set 上计算最终指标
                         - 输出 test_results.json（含按类型的详细报告）

■ 超参数
  EPOCHS=20, BATCH_SIZE=8, LR=2e-5, MAX_LEN=128, SEED=42

■ 输出文件
  logs/training_log.jsonl   → 每个 epoch 的 loss、val F1、时间戳
  logs/test_results.json    → 最终测试结果（含按类型 P/R/F1）
  model_xlmroberta/         → 最佳模型权重（可用于推理或继续训练）
================================================================================
"""

import json
import os
import time
import numpy as np
from datetime import datetime
from collections import defaultdict


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME   = "xlm-roberta-base"
DATA_DIR  = "/content/drive/MyDrive/Prima/ner_data"
LOG_DIR   = "/content/drive/MyDrive/Prima/run2/xlmroberta_article/logs"
MODEL_DIR = "/content/drive/MyDrive/Prima/run2/xlmroberta_article/model"

# ── Paramètres article (Torres Aguilar 2022) ─────────────────────────────
EPOCHS       = 5        # article : 5 epochs
BATCH_SIZE   = 16       # article : batch=16
LR           = 2e-5     # article : lr=2e-5 (identique à notre version)
MAX_LEN      = 250      # article : max_len=250
SEED         = 42

# BIO 标签体系：O + 每类实体的 B-/I- 标签
# O=0, B-auteur=1, I-auteur=2, B-ouvrage=3, I-ouvrage=4, ...
LABELS = ["auteur", "ouvrage", "material", "date", "etat"]
LABEL2ID = {"O": 0}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID)
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)  # = 11

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
VAL_DIR  = os.path.join(LOG_DIR, "resultat_val")
TEST_DIR = os.path.join(LOG_DIR, "resultat_test")
os.makedirs(VAL_DIR,  exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Device: {device}")
print(f"[Config] Model : {MODEL_NAME}")
print(f"[Config] Epochs: {EPOCHS}, LR: {LR}, Batch: {BATCH_SIZE}")


# ── 1. Conversion JSON → BIO ──────────────────────────────────────────────────
def json_to_bio(entries):
    """
    将 JSON 标注转换为 BIO 序列。
    输入：entries 列表，每条含 raw（原文）和 spans（标注片段列表）
    输出：[(tokens, labels), ...] 列表
      - tokens : 按空格切分的词列表
      - labels : 对应的 BIO 标签列表（如 B-auteur, I-auteur, O）
    """
    samples = []
    for entry in entries:
        raw = entry["raw"]
        spans = entry.get("spans", [])

        # 步骤1：按字符构建标签数组，初始全为 "O"
        # 每个字符位置对应一个标签，span 首字符标 B-，其余字符标 I-
        char_labels = ["O"] * len(raw)
        for span in spans:
            text  = span["text"]
            label = span["label"]
            idx   = raw.find(text)
            if idx == -1:
                continue  # 找不到则跳过（理论上不会发生）
            char_labels[idx] = f"B-{label}"           # 实体首字符
            for i in range(idx + 1, idx + len(text)):
                char_labels[i] = f"I-{label}"         # 实体内部字符

        # 步骤2：按空格切词，取每个词首字符的标签作为该词的标签
        # 这是"词级别"的 BIO 标注，与 subword tokenizer 的对齐在 NERDataset 中处理
        tokens = []
        labels = []
        pos = 0
        for word in raw.split():
            idx = raw.find(word, pos)
            if idx == -1:
                tokens.append(word)
                labels.append("O")
                pos += len(word)
                continue
            tok_label = char_labels[idx]  # 取词首字符的标签
            tokens.append(word)
            labels.append(tok_label)
            pos = idx + len(word)

        samples.append((tokens, labels))
    return samples


# ── 2. Dataset ────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    """
    PyTorch Dataset，将词级别的 BIO 序列转换为模型输入。

    核心问题：XLM-RoBERTa 使用 SentencePiece 切分 subword，
    一个词可能被切成多个 token（如 "Augustini" → "Aug", "ust", "ini"）。
    需要将词级别的 BIO 标签对齐到 subword 级别：
      - 词的第一个 subword 保留原标签
      - 后续 subword：若原标签是 B-X 则改为 I-X，否则保持不变
      - 特殊 token（[CLS], [SEP], [PAD]）标为 -100（loss 计算时忽略）
    """
    def __init__(self, samples, tokenizer, max_len):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens, labels = self.samples[idx]

        # subword tokenization，is_split_into_words=True 告知 tokenizer 输入已切词
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        # word_ids()：每个 subword token 对应原来第几个词（None 表示特殊 token）
        word_ids    = encoding.word_ids()
        label_ids   = []
        prev_word   = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)          # 特殊 token，loss 中忽略
            elif wid != prev_word:
                label_ids.append(LABEL2ID[labels[wid]])   # 词的第一个 subword
            else:
                # 同一个词的后续 subword：B- 改为 I-，其余不变
                l = labels[wid]
                if l.startswith("B-"):
                    l = "I-" + l[2:]
                label_ids.append(LABEL2ID.get(l, 0))
            prev_word = wid

        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(label_ids, dtype=torch.long),
        }


# ── 3. Évaluation seqeval ────────────────────────────────────────────────────
def bio_to_spans(tokens, bio):
    """Convertit une séquence BIO en liste de spans {text, label}."""
    spans = []
    cur_text, cur_label = [], None
    for tok, tag in zip(tokens, bio):
        if tag.startswith("B-"):
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text  = [tok]
            cur_label = tag[2:]
        elif tag.startswith("I-") and cur_text:
            cur_text.append(tok)
        else:
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text, cur_label = [], None
    if cur_text:
        spans.append({"text": " ".join(cur_text), "label": cur_label})
    return spans


def evaluate(model, loader, split_name="val", entries=None):
    """
    Évaluation seqeval (entité stricte) + détails par entrée optionnels.
    Retourne : {f1, precision, recall, report, details}
    """
    model.eval()
    all_preds = []
    all_trues = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            lab       = batch["labels"].to(device)
            outputs   = model(input_ids=input_ids, attention_mask=attn_mask)
            logits    = outputs.logits
            preds     = torch.argmax(logits, dim=-1)

            for pred_seq, true_seq in zip(preds.cpu().tolist(), lab.cpu().tolist()):
                p_tags = []
                t_tags = []
                for p, t in zip(pred_seq, true_seq):
                    if t == -100:
                        continue
                    p_tags.append(ID2LABEL[p])
                    t_tags.append(ID2LABEL[t])
                all_preds.append(p_tags)
                all_trues.append(t_tags)

    report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
    f1  = f1_score(all_trues, all_preds, zero_division=0)
    pre = precision_score(all_trues, all_preds, zero_division=0)
    rec = recall_score(all_trues, all_preds, zero_division=0)
    print(f"  [{split_name}] F1={f1:.4f}  P={pre:.4f}  R={rec:.4f}")
    print(classification_report(all_trues, all_preds, zero_division=0))

    details = []
    if entries is not None:
        for entry, gold_bio, pred_bio in zip(entries, all_trues, all_preds):
            tokens = entry["raw"].split()
            details.append({
                "entry_id":   entry["entry_id"],
                "raw":        entry["raw"],
                "gold_spans": bio_to_spans(tokens, gold_bio),
                "pred_spans": bio_to_spans(tokens, pred_bio),
            })

    return {"f1": f1, "precision": pre, "recall": rec,
            "report": report, "details": details}


# ── 4. Chargement des données ─────────────────────────────────────────────────
print("\n[1/5] Chargement des données...")
for split in ["train", "val", "test"]:
    path = os.path.join(DATA_DIR, f"{split}.json")
    with open(path, encoding="utf-8") as f:
        globals()[f"{split}_entries"] = json.load(f)
    print(f"  {split}: {len(globals()[f'{split}_entries'])} entrées")

train_samples = json_to_bio(train_entries)
val_samples   = json_to_bio(val_entries)
test_samples  = json_to_bio(test_entries)
print(f"  BIO → train:{len(train_samples)}, val:{len(val_samples)}, test:{len(test_samples)}")


# ── 5. Tokenizer & Dataset ────────────────────────────────────────────────────
print("\n[2/5] Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = NERDataset(train_samples, tokenizer, MAX_LEN)
val_dataset   = NERDataset(val_samples,   tokenizer, MAX_LEN)
test_dataset  = NERDataset(test_samples,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)


# ── 6. Modèle ─────────────────────────────────────────────────────────────────
print("\n[3/5] Chargement du modèle...")
# 加载 xlm-roberta-base 并替换分类头：
# 原始模型输出 768 维向量，分类头将其映射到 NUM_LABELS=11 个标签
# （O + 5类×2 = 11：O, B-auteur, I-auteur, B-ouvrage, ... B-etat, I-etat）
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

# AdamW：Adam + weight decay，适合 Transformer fine-tuning
optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
# 线性 warmup：前10%的步骤学习率从0线性增加到LR，之后线性衰减到0
# 避免训练初期大学习率破坏预训练权重
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)


# ── 7. Entraînement ───────────────────────────────────────────────────────────
print(f"\n[4/5] Entraînement ({EPOCHS} epochs)...")
log_path = os.path.join(LOG_DIR, "training_log.jsonl")
best_f1  = 0.0
training_start = time.time()

with open(log_path, "w", encoding="utf-8") as logf:
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels    = batch["labels"].to(device)

            # 前向传播：模型自动计算 cross-entropy loss（-100 位置被忽略）
            outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            loss    = outputs.loss
            total_loss += loss.item()

            # 反向传播
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 梯度裁剪防止梯度爆炸
            optimizer.step()
            scheduler.step()   # 线性 warmup + 衰减学习率
            optimizer.zero_grad()

        avg_loss   = total_loss / len(train_loader)
        epoch_time = time.time() - epoch_start
        timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        print(f"\nEpoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  time={epoch_time:.1f}s  [{timestamp}]")
        val_metrics = evaluate(model, val_loader, "val")

        # 保存 val F1 最高的模型（而非最后一个 epoch）
        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            model.save_pretrained(MODEL_DIR)
            tokenizer.save_pretrained(MODEL_DIR)
            print(f"  ✓ Meilleur modèle sauvegardé (F1={best_f1:.4f})")

        # 每个 epoch 写一行 JSON 日志（含时间戳，便于追踪训练进度）
        log_entry = {
            "epoch":         epoch,
            "timestamp":     timestamp,          # 该 epoch 结束的时刻
            "duration_s":    round(epoch_time, 2),
            "train_loss":    round(avg_loss, 6),
            "val_f1":        round(val_metrics["f1"], 6),
            "val_precision": round(val_metrics["precision"], 6),
            "val_recall":    round(val_metrics["recall"], 6),
        }
        logf.write(json.dumps(log_entry, ensure_ascii=False) + "\n")
        logf.flush()   # 实时写入，不等训练结束

total_time = time.time() - training_start
print(f"\nEntraînement terminé en {total_time/60:.1f} min | Meilleur val F1 = {best_f1:.4f}")


# ── 8. Évaluation finale sur val + test ──────────────────────────────────────
print("\n[5/5] Évaluation sur val et test sets...")
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
model.to(device)
val_metrics  = evaluate(model, val_loader,  "val",  entries=val_entries)
test_metrics = evaluate(model, test_loader, "test", entries=test_entries)

# Sauvegarder les résultats JSON
results = {
    "model":       MODEL_NAME,
    "note":        "Paramètres article Torres Aguilar 2022 : epochs=5, lr=2e-5, batch=16, max_len=250",
    "epochs":      EPOCHS,
    "lr":          LR,
    "batch_size":  BATCH_SIZE,
    "max_len":     MAX_LEN,
    "seed":        SEED,
    "best_val_f1": round(best_f1, 6),
    "val_metrics": {
        "f1":        round(val_metrics["f1"], 6),
        "precision": round(val_metrics["precision"], 6),
        "recall":    round(val_metrics["recall"], 6),
    },
    "val_report_by_label":  val_metrics["report"],
    "val_details":          val_metrics["details"],
    "test_metrics": {
        "f1":        round(test_metrics["f1"], 6),
        "precision": round(test_metrics["precision"], 6),
        "recall":    round(test_metrics["recall"], 6),
    },
    "test_report_by_label": test_metrics["report"],
    "test_details":         test_metrics["details"],
    "total_training_time_min": round(total_time / 60, 2),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

with open(os.path.join(LOG_DIR, "test_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)

# ── Rapport Markdown ──────────────────────────────────────────────────────────
LABELS_ORDER = ["auteur", "ouvrage", "material", "date", "etat"]

def error_type(gold_spans, pred_spans):
    gold_set = {(s["text"], s["label"]) for s in gold_spans}
    pred_set = {(s["text"], s["label"]) for s in pred_spans}
    if not gold_set and not pred_set:
        return "correct_vide"
    if gold_set == pred_set:
        return "correct_exact"
    if gold_set and not pred_set:
        return "faux_negatif"
    if pred_set and not gold_set:
        return "faux_positif"
    return "erreur_partielle"

def make_markdown_report(split_name, metrics, details, report, out_dir=None):
    lines = []
    lines.append(f"# Rapport d'analyse — XLM-RoBERTa ({split_name.upper()})\n")
    lines.append(f"**Paramètres** : epochs={EPOCHS}, lr={LR}, batch={BATCH_SIZE}, max_len={MAX_LEN}  ")
    lines.append(f"**Référence** : Torres Aguilar (2022), LREC LT4HALA  ")
    lines.append(f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    lines.append("---\n")
    lines.append("## Matrice de résultats (seqeval — entité stricte)\n")
    lines.append("```")
    lines.append(f"{'Label':<15} {'P':>8} {'R':>8} {'F1':>8} {'Support':>9}")
    lines.append("-" * 52)
    for lab in LABELS_ORDER:
        r = report.get(lab, {})
        lines.append(f"{lab:<15} {r.get('precision',0):>8.3f} {r.get('recall',0):>8.3f} "
                     f"{r.get('f1-score',0):>8.3f} {int(r.get('support',0)):>9}")
    lines.append("-" * 52)
    for avg in ["micro avg", "macro avg", "weighted avg"]:
        r = report.get(avg, {})
        lines.append(f"{avg:<15} {r.get('precision',0):>8.3f} {r.get('recall',0):>8.3f} "
                     f"{r.get('f1-score',0):>8.3f} {int(r.get('support',0)):>9}")
    lines.append("```\n")
    lines.append(f"- **F1** = {metrics['f1']:.4f}  **P** = {metrics['precision']:.4f}  **R** = {metrics['recall']:.4f}\n")
    lines.append("---\n")
    lines.append("## Exemples détaillés par type d'erreur\n")
    grouped = {"correct_exact": [], "correct_vide": [],
               "faux_negatif": [], "faux_positif": [], "erreur_partielle": []}
    for d in details:
        t = error_type(d["gold_spans"], d["pred_spans"])
        grouped[t].append(d)
    labels_fr = {
        "correct_exact":   "✅ Correct — correspondance exacte",
        "correct_vide":    "✅ Correct — aucune entité attendue ni prédite",
        "faux_negatif":    "❌ Faux négatif — entité non détectée",
        "faux_positif":    "⚠️ Faux positif — entité prédite sans gold",
        "erreur_partielle":"〰️ Erreur partielle — détection incomplète ou mauvais label",
    }
    for key, title in labels_fr.items():
        entries_g = grouped[key]
        if not entries_g:
            continue
        lines.append(f"### {title} ({len(entries_g)} entrées)\n")
        for d in entries_g:
            gold_str = ", ".join(f"{s['label']}({s['text']})" for s in d["gold_spans"]) or "—"
            pred_str = ", ".join(f"{s['label']}({s['text']})" for s in d["pred_spans"]) or "—"
            lines.append(f"**#{d['entry_id']}** `{d['raw'][:90]}`")
            lines.append(f"- Gold : {gold_str}")
            lines.append(f"- Préd : {pred_str}\n")
    report_content = "\n".join(lines) + "\n"
    if out_dir:
        md_path = os.path.join(out_dir, f"analysis_{split_name}.md")
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        print(f"  → Rapport {split_name} : {md_path}")
        json_path = os.path.join(out_dir, f"results_{split_name}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({"split": split_name, "metrics": metrics,
                       "report_by_label": report, "details": details},
                      f, ensure_ascii=False, indent=2, cls=NumpyEncoder)
        print(f"  → JSON {split_name}    : {json_path}")
    return report_content


make_markdown_report("val",  val_metrics,  val_metrics["details"],  val_metrics["report"],  VAL_DIR)
make_markdown_report("test", test_metrics, test_metrics["details"], test_metrics["report"], TEST_DIR)

# Rapport global (val + test ensemble)
md_lines = []
md_lines.append("# Rapport d'analyse — XLM-RoBERTa (paramètres article)\n")
md_lines.append(f"**Paramètres** : epochs={EPOCHS}, lr={LR}, batch={BATCH_SIZE}, max_len={MAX_LEN}  ")
md_lines.append(f"**Référence** : Torres Aguilar (2022), LREC LT4HALA  ")
md_lines.append(f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
md_lines.append("---\n")
md_lines.append(make_markdown_report("val",  val_metrics,  val_metrics["details"],  val_metrics["report"]))
md_lines.append("---\n")
md_lines.append(make_markdown_report("test", test_metrics, test_metrics["details"], test_metrics["report"]))

md_path = os.path.join(LOG_DIR, "analysis_article_params.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines) + "\n")

print(f"\n[Done] Val  F1={val_metrics['f1']:.4f}")
print(f"       Test F1={test_metrics['f1']:.4f}")
print(f"       Rapport global → {md_path}")
print(f"       Val            → {VAL_DIR}/")
print(f"       Test           → {TEST_DIR}/")
print(f"       Modèle         → {MODEL_DIR}/")


[Config] Device: cuda
[Config] Model : xlm-roberta-base
[Config] Epochs: 5, LR: 2e-05, Batch: 16

[1/5] Chargement des données...
  train: 70 entrées
  val: 20 entrées
  test: 10 entrées
  BIO → train:70, val:20, test:10

[2/5] Chargement du tokenizer...

[3/5] Chargement du modèle...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[4/5] Entraînement (5 epochs)...

Epoch 1/5  loss=2.2238  time=3.5s  [2026-06-07 22:12:50]
  [val] F1=0.0000  P=0.0000  R=0.0000
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00        17
        date       0.00      0.00      0.00        20
        etat       0.00      0.00      0.00         6
    material       0.00      0.00      0.00        21
     ouvrage       0.00      0.00      0.00        23

   micro avg       0.00      0.00      0.00        87
   macro avg       0.00      0.00      0.00        87
weighted avg       0.00      0.00      0.00        87


Epoch 2/5  loss=2.0155  time=3.5s  [2026-06-07 22:12:54]
  [val] F1=0.0000  P=0.0000  R=0.0000
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00        17
        date       0.00      0.00      0.00        20
        etat       0.00      0.00      0.00         6
    material       0.00      0.00      0.00        21
     ouvrage  

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  ✓ Meilleur modèle sauvegardé (F1=0.0282)

Epoch 5/5  loss=1.5519  time=3.4s  [2026-06-07 22:13:24]
  [val] F1=0.0474  P=0.0403  R=0.0575
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00        17
        date       0.00      0.00      0.00        20
        etat       0.00      0.00      0.00         6
    material       0.09      0.19      0.12        21
     ouvrage       0.02      0.04      0.03        23

   micro avg       0.04      0.06      0.05        87
   macro avg       0.02      0.05      0.03        87
weighted avg       0.03      0.06      0.04        87



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Meilleur modèle sauvegardé (F1=0.0474)

Entraînement terminé en 1.5 min | Meilleur val F1 = 0.0474

[5/5] Évaluation sur val et test sets...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [val] F1=0.0474  P=0.0403  R=0.0575
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00        17
        date       0.00      0.00      0.00        20
        etat       0.00      0.00      0.00         6
    material       0.09      0.19      0.12        21
     ouvrage       0.02      0.04      0.03        23

   micro avg       0.04      0.06      0.05        87
   macro avg       0.02      0.05      0.03        87
weighted avg       0.03      0.06      0.04        87

  [test] F1=0.0500  P=0.0405  R=0.0652
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00        10
        date       0.00      0.00      0.00        10
        etat       0.00      0.00      0.00         4
    material       0.12      0.27      0.17        11
     ouvrage       0.00      0.00      0.00        11

   micro avg       0.04      0.07      0.05        46
   macro avg       0.03      0.05      0.03        46


## Cell 3 — BiLSTM-CRF (nos paramètres)
`epochs=50, batch=8, lr=0.001`  →  `Prima/run2/bilstm/`

In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL — BiLSTM-CRF — NOS paramètres (epochs=50, batch=8, lr=0.001)
# Données  : /content/drive/MyDrive/Prima/ner_data/
# Sorties  : /content/drive/MyDrive/Prima/run2/bilstm/
# ═══════════════════════════════════════════════════════════════

# -*- coding: utf-8 -*-
"""
prima_bilstm_crf.py
===================
Baseline BiLSTM-CRF pour la NER sur les entrées du catalogue Riccardiana.

Labels : auteur, ouvrage, material, date, etat  (schéma BIO)
Données : train.json / val.json / test.json

Sorties dans ./logs/ et ./model_bilstm/ :
  - logs/training_log.jsonl   → loss + F1 + temps par epoch
  - logs/test_results.json    → métriques finales sur test
  - model_bilstm/model.pt     → meilleurs poids du modèle

Usage (dans ~/prima_ner/bilstm_crf/) :
    python3 prima_bilstm_crf.py

Dépendances :
    pip install torch seqeval --use-pep517

================================================================================
【代码逻辑与结构说明】
================================================================================

■ 任务目标
  与 XLM-RoBERTa 相同的 NER 任务，作为 baseline 对比系统。
  BiLSTM-CRF 是经典的序列标注模型，不依赖预训练语言模型。

■ 模型架构
  词嵌入层（Embedding）
    → 双向 LSTM（BiLSTM）：同时从左到右和从右到左读取序列，
      捕捉每个词的上下文信息，输出维度 = 2 × hidden_dim
    → 线性层：将 BiLSTM 输出映射到标签空间（NUM_LABELS 维）
    → CRF 层：对标签序列建模，学习标签之间的转移概率
              （如 I-auteur 不能跟在 B-ouvrage 后面）
              使用 Viterbi 算法解码最优标签序列

■ 特征
  - 词级别（word-level）：每个词对应一个 embedding 向量
  - 词表从训练集构建，OOV（未登录词）用 <UNK> 处理
  - 无预训练词向量（从零开始训练），这是与 XLM-RoBERTa 的主要区别

■ 数据处理
  - 同 XLM-RoBERTa：JSON → BIO 转换（字符级对齐）
  - padding 到批次最大长度（动态 padding，不固定 MAX_LEN）

■ 代码结构（按执行顺序）
  1. json_to_bio()      : JSON 标注 → BIO 词序列（同 XLM-RoBERTa）
  2. 构建词表/标签表    : 从训练集统计所有词，构建 word2id/id2word
  3. NERDataset         : 将词序列转为 id 序列，pad 到批次最长
  4. BiLSTM_CRF 类      : 模型定义（Embedding + BiLSTM + Linear + CRF）
  5. CRF 实现           : neg_log_likelihood（训练loss）+ viterbi_decode（推理）
  6. bio_to_spans()     : 将 BIO 序列还原为实体 span 列表 {text, label}
  7. evaluate()         : seqeval 实体级别 P/R/F1
                          - 可选 entries 参数：记录每条 entry 的 gold/pred span 明细
  8. 训练循环           : 每 epoch 记录 loss/F1/时间戳，保存最佳模型
  9. 测试评估           : 加载最佳模型，同时在 val 和 test 上计算最终指标
  10. make_markdown_report() : 生成按错误类型分组的 Markdown 报告
                          - correct_exact / correct_vide / faux_negatif /
                            faux_positif / erreur_partielle
                          - val 和 test 各有独立目录（resultat_val / resultat_test）

■ 超参数
  EPOCHS=50, BATCH_SIZE=8, LR=0.001, EMBED_DIM=64, HIDDEN_DIM=128, SEED=42
  （BiLSTM 收敛慢于 Transformer，需要更多 epoch）

■ 与 XLM-RoBERTa 的主要区别
  - 无预训练：从零开始学习词表示，需要更多数据才能达到同等效果
  - 速度快：模型小，训练快（无需 GPU 也可运行）
  - 泛化弱：对未见词（OOV）处理能力差
  - 这正是 baseline 的意义：展示预训练模型带来的提升幅度

■ 输出文件
  logs/training_log.jsonl          → 每个 epoch 的 loss、val F1、时间戳
  logs/test_results.json           → val+test 完整结果（含 entry 明细）
  logs/analysis_report.md          → val+test 合并 Markdown 报告
  logs/resultat_val/               → val 独立目录（JSON + Markdown）
  logs/resultat_test/              → test 独立目录（JSON + Markdown）
  model_bilstm/model.pt            → 最佳模型权重
================================================================================
"""

import json
import os
import time
import random
import re
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR  = "/content/drive/MyDrive/Prima/ner_data"
LOG_DIR   = "/content/drive/MyDrive/Prima/run2/bilstm/logs"
MODEL_DIR = "/content/drive/MyDrive/Prima/run2/bilstm/model"

EPOCHS     = 50       # nos paramètres
BATCH_SIZE = 8
LR         = 0.001    # nos paramètres
EMBED_DIM  = 64       # 词嵌入维度
HIDDEN_DIM = 128      # BiLSTM 隐藏层维度（单向），双向输出为 256
DROPOUT    = 0.3      # Dropout 防止过拟合
SEED       = 42

# BIO 标签体系（与 XLM-RoBERTa 完全一致，便于对比）
LABELS = ["auteur", "ouvrage", "material", "date", "etat"]
LABEL2ID = {"O": 0, "PAD": -1}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID) - 1
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID) - 1
# 重建干净的标签映射（去掉 PAD）
LABEL2ID = {"O": 0}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID)
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
VAL_DIR  = os.path.join(LOG_DIR, "resultat_val")
TEST_DIR = os.path.join(LOG_DIR, "resultat_test")
os.makedirs(VAL_DIR,  exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Device    : {device}")
print(f"[Config] Epochs    : {EPOCHS}, LR: {LR}, Batch: {BATCH_SIZE}")
print(f"[Config] Embed_dim : {EMBED_DIM}, Hidden_dim: {HIDDEN_DIM}")


# ── 1. Conversion JSON → BIO ──────────────────────────────────────────────────
def json_to_bio(entries):
    """
    JSON 标注 → BIO 词序列（与 XLM-RoBERTa 脚本完全一致）
    按字符位置构建 char_labels，然后按空格切词取词首字符标签。
    """
    samples = []
    for entry in entries:
        raw   = entry["raw"]
        spans = entry.get("spans", [])

        char_labels = ["O"] * len(raw)
        for span in spans:
            text  = span["text"]
            label = span["label"]
            idx   = raw.find(text)
            if idx == -1:
                continue
            char_labels[idx] = f"B-{label}"
            for i in range(idx + 1, idx + len(text)):
                char_labels[i] = f"I-{label}"

        tokens = []
        labels = []
        pos = 0
        for word in raw.split():
            idx = raw.find(word, pos)
            if idx == -1:
                tokens.append(word); labels.append("O")
                pos += len(word); continue
            tokens.append(word)
            labels.append(char_labels[idx])
            pos = idx + len(word)

        samples.append((tokens, labels))
    return samples


# ── 2. Construire le vocabulaire ──────────────────────────────────────────────
def build_vocab(train_samples):
    """
    从训练集构建词表。
    特殊符号：<PAD>（填充）, <UNK>（未知词/OOV）
    """
    word2id = {"<PAD>": 0, "<UNK>": 1}
    for tokens, _ in train_samples:
        for w in tokens:
            if w not in word2id:
                word2id[w] = len(word2id)
    return word2id


# ── 3. Dataset ────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    """
    将词序列转为 id 序列。
    未知词（测试集中未在训练集出现的词）映射到 <UNK>。
    """
    def __init__(self, samples, word2id):
        self.data    = samples
        self.word2id = word2id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]
        token_ids = [self.word2id.get(w, self.word2id["<UNK>"]) for w in tokens]
        label_ids = [LABEL2ID[l] for l in labels]
        return torch.tensor(token_ids, dtype=torch.long), \
               torch.tensor(label_ids, dtype=torch.long)


def collate_fn(batch):
    """
    动态 padding：将一个 batch 内的序列 pad 到该 batch 最大长度。
    比固定 MAX_LEN 更高效。
    """
    token_seqs, label_seqs = zip(*batch)
    token_padded = pad_sequence(token_seqs, batch_first=True, padding_value=0)
    label_padded = pad_sequence(label_seqs, batch_first=True, padding_value=-1)
    mask = (token_padded != 0)   # True = 真实 token，False = PAD
    return token_padded, label_padded, mask


# ── 4. Modèle BiLSTM-CRF ────────────────────────────────────────────────────
class BiLSTM_CRF(nn.Module):
    """
    BiLSTM-CRF 序列标注模型。

    架构：
      Embedding → Dropout → BiLSTM → Dropout → Linear → CRF

    CRF 层维护一个 (NUM_LABELS × NUM_LABELS) 的转移矩阵，
    学习标签之间的合法转移（如禁止 I-auteur 跟在 B-date 后面）。
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, dropout):
        super().__init__()
        self.num_labels = num_labels

        # 词嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout   = nn.Dropout(dropout)

        # 双向 LSTM：输出维度 = 2 × hidden_dim
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                            bidirectional=True, num_layers=1)

        # 将 BiLSTM 输出映射到标签空间（发射分数）
        self.hidden2tag = nn.Linear(hidden_dim * 2, num_labels)

        # CRF 转移矩阵：transitions[i][j] = 从标签 j 转移到标签 i 的分数
        self.transitions = nn.Parameter(torch.randn(num_labels, num_labels))
        # 初始化：禁止转移到 O 标签的中间位置（软约束）
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def get_emissions(self, x):
        """计算发射分数（BiLSTM 输出）"""
        emb  = self.dropout(self.embedding(x))
        out, _ = self.lstm(emb)
        out  = self.dropout(out)
        return self.hidden2tag(out)   # (batch, seq_len, num_labels)

    def crf_forward(self, emissions, mask):
        """
        CRF 前向算法：计算所有可能路径的归一化因子 log Z。
        用于训练时计算 negative log-likelihood。
        """
        batch_size, seq_len, _ = emissions.shape
        # 初始化：第一个时间步的分数
        score = emissions[:, 0, :]   # (batch, num_labels)
        for t in range(1, seq_len):
            # broadcast：当前分数 + 转移分数 + 发射分数
            score_t = score.unsqueeze(2) + self.transitions.unsqueeze(0) \
                      + emissions[:, t, :].unsqueeze(1)
            score_t = torch.logsumexp(score_t, dim=1)
            # mask：PAD 位置不更新分数
            mask_t  = mask[:, t].unsqueeze(1).float()
            score   = score_t * mask_t + score * (1 - mask_t)
        return torch.logsumexp(score, dim=1)   # (batch,)

    def score_sequence(self, emissions, tags, mask):
        """
        计算给定标签序列的路径分数（发射分数 + 转移分数之和）。
        """
        batch_size, seq_len, _ = emissions.shape
        score = torch.zeros(batch_size, device=emissions.device)
        for t in range(seq_len):
            mask_t = mask[:, t].float()
            emit   = emissions[:, t, :].gather(1, tags[:, t].unsqueeze(1)).squeeze(1)
            if t > 0:
                trans = self.transitions[tags[:, t], tags[:, t-1]]
                score = score + (emit + trans) * mask_t
            else:
                score = score + emit * mask_t
        return score

    def neg_log_likelihood(self, x, tags, mask):
        """
        训练损失：负对数似然 = log Z - score(正确路径)
        """
        emissions = self.get_emissions(x)
        # 将 PAD 位置的标签（-1）替换为 0，避免索引越界
        tags_safe = tags.clone()
        tags_safe[tags_safe < 0] = 0
        forward   = self.crf_forward(emissions, mask)
        gold      = self.score_sequence(emissions, tags_safe, mask)
        return (forward - gold).mean()

    def viterbi_decode(self, x, mask):
        """
        Viterbi 算法：找到最优标签序列（推理阶段）。
        """
        emissions  = self.get_emissions(x)
        batch_size, seq_len, _ = emissions.shape

        viterbi  = emissions[:, 0, :]
        backptr  = []

        for t in range(1, seq_len):
            v_t   = viterbi.unsqueeze(2) + self.transitions.unsqueeze(0)
            best_scores, best_tags = v_t.max(dim=1)
            backptr.append(best_tags)
            mask_t  = mask[:, t].unsqueeze(1).float()
            viterbi = (best_scores + emissions[:, t, :]) * mask_t \
                      + viterbi * (1 - mask_t)

        # 回溯
        best_last = viterbi.argmax(dim=1)
        best_path = [best_last]
        for bp in reversed(backptr):
            best_last = bp.gather(1, best_last.unsqueeze(1)).squeeze(1)
            best_path.append(best_last)
        best_path.reverse()
        best_path = torch.stack(best_path, dim=1)   # (batch, seq_len)
        return best_path


# ── 5. Évaluation ─────────────────────────────────────────────────────────────
def bio_to_spans(tokens, bio):
    """
    将 BIO 序列还原为实体 span 列表。
    输入：tokens（词列表）, bio（对应的 BIO 标签列表）
    输出：[{"text": "...", "label": "..."}, ...]
    """
    spans = []
    cur_text, cur_label = [], None
    for tok, tag in zip(tokens, bio):
        if tag.startswith("B-"):
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text  = [tok]
            cur_label = tag[2:]
        elif tag.startswith("I-") and cur_text:
            cur_text.append(tok)
        else:
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text, cur_label = [], None
    if cur_text:
        spans.append({"text": " ".join(cur_text), "label": cur_label})
    return spans


def evaluate(model, loader, split_name="val", entries=None):
    """
    seqeval 实体级别评估，与 XLM-RoBERTa 脚本完全一致。

    参数：
      entries : 若传入原始 JSON 条目列表，则记录每条 entry 的
                gold_spans 和 pred_spans 明细（用于 Markdown 报告）

    输出：
      - 终端打印 P/R/F1
      - 返回 dict：{f1, precision, recall, report, details}
    """
    model.eval()
    all_preds = []
    all_trues = []
    with torch.no_grad():
        for token_ids, label_ids, mask in loader:
            token_ids = token_ids.to(device)
            mask      = mask.to(device)
            preds     = model.viterbi_decode(token_ids, mask)

            for pred_seq, true_seq, m in zip(
                    preds.cpu().tolist(), label_ids.tolist(), mask.cpu().tolist()):
                p_tags = [ID2LABEL[p] for p, valid in zip(pred_seq, m) if valid]
                t_tags = [ID2LABEL[t] for t, valid in zip(true_seq, m) if valid]
                all_preds.append(p_tags)
                all_trues.append(t_tags)

    f1  = f1_score(all_trues, all_preds, zero_division=0)
    pre = precision_score(all_trues, all_preds, zero_division=0)
    rec = recall_score(all_trues, all_preds, zero_division=0)
    report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
    print(f"  [{split_name}] F1={f1:.4f}  P={pre:.4f}  R={rec:.4f}")
    print(classification_report(all_trues, all_preds, zero_division=0))

    # entry 级别明细（用于 Markdown 报告的错误分析）
    details = []
    if entries is not None:
        for entry, gold_bio, pred_bio in zip(entries, all_trues, all_preds):
            tokens = entry["raw"].split()
            details.append({
                "entry_id":   entry["entry_id"],
                "raw":        entry["raw"],
                "gold_spans": bio_to_spans(tokens, gold_bio),
                "pred_spans": bio_to_spans(tokens, pred_bio),
            })

    return {"f1": f1, "precision": pre, "recall": rec,
            "report": report, "details": details}


# ── 6. Chargement des données ─────────────────────────────────────────────────
print("\n[1/5] Chargement des données...")
splits = {}
for split in ["train", "val", "test"]:
    with open(os.path.join(DATA_DIR, f"{split}.json"), encoding="utf-8") as f:
        splits[split] = json.load(f)
    print(f"  {split}: {len(splits[split])} entrées")

train_samples = json_to_bio(splits["train"])
val_samples   = json_to_bio(splits["val"])
test_samples  = json_to_bio(splits["test"])

# ── 7. Vocabulaire ────────────────────────────────────────────────────────────
print("\n[2/5] Construction du vocabulaire...")
word2id = build_vocab(train_samples)
print(f"  Vocab size: {len(word2id)} mots")

train_dataset = NERDataset(train_samples, word2id)
val_dataset   = NERDataset(val_samples,   word2id)
test_dataset  = NERDataset(test_samples,  word2id)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, collate_fn=collate_fn)

# ── 8. Modèle ─────────────────────────────────────────────────────────────────
print("\n[3/5] Initialisation du modèle BiLSTM-CRF...")
model = BiLSTM_CRF(
    vocab_size  = len(word2id),
    embed_dim   = EMBED_DIM,
    hidden_dim  = HIDDEN_DIM,
    num_labels  = NUM_LABELS,
    dropout     = DROPOUT,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Paramètres entraînables: {total_params:,}")

# ── 9. Entraînement ───────────────────────────────────────────────────────────
print(f"\n[4/5] Entraînement ({EPOCHS} epochs)...")
log_path = os.path.join(LOG_DIR, "training_log.jsonl")
best_f1  = 0.0
training_start = time.time()

with open(log_path, "w", encoding="utf-8") as logf:
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0

        for token_ids, label_ids, mask in train_loader:
            token_ids = token_ids.to(device)
            label_ids = label_ids.to(device)
            mask      = mask.to(device)

            # CRF 的 loss：负对数似然
            loss = model.neg_log_likelihood(token_ids, label_ids, mask)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        avg_loss   = total_loss / len(train_loader)
        epoch_time = time.time() - epoch_start
        timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        print(f"\nEpoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  "
              f"time={epoch_time:.1f}s  [{timestamp}]")
        val_metrics = evaluate(model, val_loader, "val", entries=splits["val"])

        # 保存最佳模型
        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            torch.save(model.state_dict(),
                       os.path.join(MODEL_DIR, "model.pt"))
            # 保存词表（推理时需要）
            with open(os.path.join(MODEL_DIR, "word2id.json"), "w") as f:
                json.dump(word2id, f, ensure_ascii=False)
            print(f"  ✓ Meilleur modèle sauvegardé (F1={best_f1:.4f})")

        log_entry = {
            "epoch":         epoch,
            "timestamp":     timestamp,
            "duration_s":    round(epoch_time, 2),
            "train_loss":    round(avg_loss, 6),
            "val_f1":        round(val_metrics["f1"], 6),
            "val_precision": round(val_metrics["precision"], 6),
            "val_recall":    round(val_metrics["recall"], 6),
        }
        logf.write(json.dumps(log_entry, ensure_ascii=False) + "\n")
        logf.flush()

total_time = time.time() - training_start
print(f"\nEntraînement terminé en {total_time/60:.1f} min | "
      f"Meilleur val F1 = {best_f1:.4f}")

# ── 10. Évaluation finale sur val + test ─────────────────────────────────────
print("\n[5/5] Évaluation sur val et test sets...")
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "model.pt"),
                                  map_location=device))
val_metrics  = evaluate(model, val_loader,  "val",  entries=splits["val"])
test_metrics = evaluate(model, test_loader, "test", entries=splits["test"])

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

results = {
    "model":      "BiLSTM-CRF",
    "epochs":     EPOCHS,
    "lr":         LR,
    "embed_dim":  EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "batch_size": BATCH_SIZE,
    "seed":       SEED,
    "vocab_size": len(word2id),
    "best_val_f1": round(best_f1, 6),
    "val_metrics": {
        "f1":        round(val_metrics["f1"], 6),
        "precision": round(val_metrics["precision"], 6),
        "recall":    round(val_metrics["recall"], 6),
    },
    "val_report_by_label":  val_metrics["report"],
    "val_details":          val_metrics["details"],
    "test_metrics": {
        "f1":        round(test_metrics["f1"], 6),
        "precision": round(test_metrics["precision"], 6),
        "recall":    round(test_metrics["recall"], 6),
    },
    "test_report_by_label": test_metrics["report"],
    "test_details":         test_metrics["details"],
    "total_training_time_min": round(total_time / 60, 2),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

with open(os.path.join(LOG_DIR, "test_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)

# ── Rapport Markdown ──────────────────────────────────────────────────────────
LABELS_ORDER = ["auteur", "ouvrage", "material", "date", "etat"]


def error_type(gold_spans, pred_spans):
    gold_set = {(s["text"], s["label"]) for s in gold_spans}
    pred_set = {(s["text"], s["label"]) for s in pred_spans}
    if not gold_set and not pred_set:
        return "correct_vide"
    if gold_set == pred_set:
        return "correct_exact"
    if gold_set and not pred_set:
        return "faux_negatif"
    if pred_set and not gold_set:
        return "faux_positif"
    return "erreur_partielle"


def make_markdown_report(split_name, metrics, details, report, out_dir=None):
    lines = []
    lines.append(f"# Rapport d'analyse — BiLSTM-CRF ({split_name.upper()})\n")
    lines.append(f"**Paramètres** : epochs={EPOCHS}, lr={LR}, batch={BATCH_SIZE}, "
                 f"embed_dim={EMBED_DIM}, hidden_dim={HIDDEN_DIM}  ")
    lines.append(f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    lines.append("---\n")
    lines.append("## Matrice de résultats (seqeval — entité stricte)\n")
    lines.append("```")
    lines.append(f"{'Label':<15} {'P':>8} {'R':>8} {'F1':>8} {'Support':>9}")
    lines.append("-" * 52)
    for lab in LABELS_ORDER:
        r = report.get(lab, {})
        lines.append(f"{lab:<15} {r.get('precision',0):>8.3f} {r.get('recall',0):>8.3f} "
                     f"{r.get('f1-score',0):>8.3f} {int(r.get('support',0)):>9}")
    lines.append("-" * 52)
    for avg in ["micro avg", "macro avg", "weighted avg"]:
        r = report.get(avg, {})
        lines.append(f"{avg:<15} {r.get('precision',0):>8.3f} {r.get('recall',0):>8.3f} "
                     f"{r.get('f1-score',0):>8.3f} {int(r.get('support',0)):>9}")
    lines.append("```\n")
    lines.append(f"- **F1** = {metrics['f1']:.4f}  **P** = {metrics['precision']:.4f}  **R** = {metrics['recall']:.4f}\n")
    lines.append("---\n")
    lines.append("## Exemples détaillés par type d'erreur\n")
    grouped = {"correct_exact": [], "correct_vide": [],
               "faux_negatif": [], "faux_positif": [], "erreur_partielle": []}
    for d in details:
        t = error_type(d["gold_spans"], d["pred_spans"])
        grouped[t].append(d)
    labels_fr = {
        "correct_exact":    "✅ Correct — correspondance exacte",
        "correct_vide":     "✅ Correct — aucune entité attendue ni prédite",
        "faux_negatif":     "❌ Faux négatif — entité non détectée",
        "faux_positif":     "⚠️ Faux positif — entité prédite sans gold",
        "erreur_partielle": "〰️ Erreur partielle — détection incomplète ou mauvais label",
    }
    for key, title in labels_fr.items():
        entries_g = grouped[key]
        if not entries_g:
            continue
        lines.append(f"### {title} ({len(entries_g)} entrées)\n")
        for d in entries_g:
            gold_str = ", ".join(f"{s['label']}({s['text']})" for s in d["gold_spans"]) or "—"
            pred_str = ", ".join(f"{s['label']}({s['text']})" for s in d["pred_spans"]) or "—"
            lines.append(f"**#{d['entry_id']}** `{d['raw'][:90]}`")
            lines.append(f"- Gold : {gold_str}")
            lines.append(f"- Préd : {pred_str}\n")
    report_content = "\n".join(lines) + "\n"
    if out_dir:
        md_path = os.path.join(out_dir, f"analysis_{split_name}.md")
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        print(f"  → Rapport {split_name} : {md_path}")
        json_path = os.path.join(out_dir, f"results_{split_name}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({"split": split_name, "metrics": metrics,
                       "report_by_label": report, "details": details},
                      f, ensure_ascii=False, indent=2, cls=NumpyEncoder)
        print(f"  → JSON {split_name}    : {json_path}")
    return report_content


make_markdown_report("val",  val_metrics,  val_metrics["details"],  val_metrics["report"],  VAL_DIR)
make_markdown_report("test", test_metrics, test_metrics["details"], test_metrics["report"], TEST_DIR)

# Rapport global (val + test ensemble)
md_lines = []
md_lines.append("# Rapport d'analyse — BiLSTM-CRF (nos paramètres)\n")
md_lines.append(f"**Paramètres** : epochs={EPOCHS}, lr={LR}, batch={BATCH_SIZE}, "
                f"embed_dim={EMBED_DIM}, hidden_dim={HIDDEN_DIM}  ")
md_lines.append(f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
md_lines.append("---\n")
md_lines.append(make_markdown_report("val",  val_metrics,  val_metrics["details"],  val_metrics["report"]))
md_lines.append("---\n")
md_lines.append(make_markdown_report("test", test_metrics, test_metrics["details"], test_metrics["report"]))

md_path = os.path.join(LOG_DIR, "analysis_report.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines) + "\n")

print(f"\n[Done] Val  F1={val_metrics['f1']:.4f}")
print(f"       Test F1={test_metrics['f1']:.4f}")
print(f"       Rapport global → {md_path}")
print(f"       Val            → {VAL_DIR}/")
print(f"       Test           → {TEST_DIR}/")
print(f"       Modèle         → {MODEL_DIR}/model.pt")


[Config] Device    : cuda
[Config] Epochs    : 50, LR: 0.001, Batch: 8
[Config] Embed_dim : 64, Hidden_dim: 128

[1/5] Chargement des données...
  train: 70 entrées
  val: 20 entrées
  test: 10 entrées

[2/5] Construction du vocabulaire...
  Vocab size: 509 mots

[3/5] Initialisation du modèle BiLSTM-CRF...
  Paramètres entraînables: 234,180

[4/5] Entraînement (50 epochs)...

Epoch 1/50  loss=34.4221  time=2.3s  [2026-06-07 22:34:48]
  [val] F1=0.2367  P=0.2439  R=0.2299
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00        17
        date       0.53      0.45      0.49        20
        etat       0.00      0.00      0.00         6
    material       0.18      0.43      0.25        21
     ouvrage       0.13      0.09      0.11        23

   micro avg       0.24      0.23      0.24        87
   macro avg       0.17      0.19      0.17        87
weighted avg       0.20      0.23      0.20        87

  ✓ Meilleur modèle sauvegardé (F1

## Cell 4 — BiLSTM-CRF (paramètres article)
`epochs=5, batch=16, lr=0.01`  →  `Prima/run2/bilstm_article/`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL — BiLSTM-CRF — paramètres ARTICLE (epochs=5, batch=16, lr=0.01)
# Données  : /content/drive/MyDrive/Prima/ner_data/
# Sorties  : /content/drive/MyDrive/Prima/run2/bilstm_article/
# ═══════════════════════════════════════════════════════════════

# -*- coding: utf-8 -*-
"""
prima_bilstm_crf.py
===================
Baseline BiLSTM-CRF pour la NER sur les entrées du catalogue Riccardiana.

Labels : auteur, ouvrage, material, date, etat  (schéma BIO)
Données : train.json / val.json / test.json

Sorties dans ./logs/ et ./model_bilstm/ :
  - logs/training_log.jsonl   → loss + F1 + temps par epoch
  - logs/test_results.json    → métriques finales sur test
  - model_bilstm/model.pt     → meilleurs poids du modèle

Usage (dans ~/prima_ner/bilstm_crf/) :
    python3 prima_bilstm_crf.py

Dépendances :
    pip install torch seqeval --use-pep517

================================================================================
【代码逻辑与结构说明】
================================================================================

■ 任务目标
  与 XLM-RoBERTa 相同的 NER 任务，作为 baseline 对比系统。
  BiLSTM-CRF 是经典的序列标注模型，不依赖预训练语言模型。

■ 模型架构
  词嵌入层（Embedding）
    → 双向 LSTM（BiLSTM）：同时从左到右和从右到左读取序列，
      捕捉每个词的上下文信息，输出维度 = 2 × hidden_dim
    → 线性层：将 BiLSTM 输出映射到标签空间（NUM_LABELS 维）
    → CRF 层：对标签序列建模，学习标签之间的转移概率
              （如 I-auteur 不能跟在 B-ouvrage 后面）
              使用 Viterbi 算法解码最优标签序列

■ 特征
  - 词级别（word-level）：每个词对应一个 embedding 向量
  - 词表从训练集构建，OOV（未登录词）用 <UNK> 处理
  - 无预训练词向量（从零开始训练），这是与 XLM-RoBERTa 的主要区别

■ 数据处理
  - 同 XLM-RoBERTa：JSON → BIO 转换（字符级对齐）
  - padding 到批次最大长度（动态 padding，不固定 MAX_LEN）

■ 代码结构（按执行顺序）
  1. json_to_bio()      : JSON 标注 → BIO 词序列（同 XLM-RoBERTa）
  2. 构建词表/标签表    : 从训练集统计所有词，构建 word2id/id2word
  3. NERDataset         : 将词序列转为 id 序列，pad 到批次最长
  4. BiLSTM_CRF 类      : 模型定义（Embedding + BiLSTM + Linear + CRF）
  5. CRF 实现           : neg_log_likelihood（训练loss）+ viterbi_decode（推理）
  6. evaluate()         : seqeval 实体级别 P/R/F1
  7. 训练循环           : 每 epoch 记录 loss/F1/时间戳，保存最佳模型
  8. 测试评估           : 加载最佳模型，在 test set 上计算最终指标

■ 超参数
  EPOCHS=50, BATCH_SIZE=8, LR=0.001, EMBED_DIM=64, HIDDEN_DIM=128, SEED=42
  （BiLSTM 收敛慢于 Transformer，需要更多 epoch）

■ 与 XLM-RoBERTa 的主要区别
  - 无预训练：从零开始学习词表示，需要更多数据才能达到同等效果
  - 速度快：模型小，训练快（无需 GPU 也可运行）
  - 泛化弱：对未见词（OOV）处理能力差
  - 这正是 baseline 的意义：展示预训练模型带来的提升幅度
================================================================================
"""

import json
import os
import time
import random
import re
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR  = "/content/drive/MyDrive/Prima/ner_data"
LOG_DIR   = "/content/drive/MyDrive/Prima/run2/bilstm_article/logs"
MODEL_DIR = "/content/drive/MyDrive/Prima/run2/bilstm_article/model"

# ── Paramètres article (Torres Aguilar 2022 — BiLSTM-CRF) ───────────────
# Note: article utilise Flair+FastText ; nous adaptons les hyperparamètres
# compatibles : batch=16, lr proche (1e-2 à 5e-3 dans article)
EPOCHS     = 5        # article : 5 epochs (même valeur)
BATCH_SIZE = 16       # article : batch=16
LR         = 0.01     # article : lr entre 1e-2 et 2e-2 pour BiLSTM
EMBED_DIM  = 64
HIDDEN_DIM = 128
DROPOUT    = 0.3
SEED       = 42

# BIO 标签体系（与 XLM-RoBERTa 完全一致，便于对比）
LABELS = ["auteur", "ouvrage", "material", "date", "etat"]
LABEL2ID = {"O": 0, "PAD": -1}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID) - 1
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID) - 1
# 重建干净的标签映射（去掉 PAD）
LABEL2ID = {"O": 0}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID)
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
VAL_DIR  = os.path.join(LOG_DIR, "resultat_val")
TEST_DIR = os.path.join(LOG_DIR, "resultat_test")
os.makedirs(VAL_DIR,  exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Device    : {device}")
print(f"[Config] Epochs    : {EPOCHS}, LR: {LR}, Batch: {BATCH_SIZE}")
print(f"[Config] Embed_dim : {EMBED_DIM}, Hidden_dim: {HIDDEN_DIM}")


# ── 1. Conversion JSON → BIO ──────────────────────────────────────────────────
def json_to_bio(entries):
    """
    JSON 标注 → BIO 词序列（与 XLM-RoBERTa 脚本完全一致）
    按字符位置构建 char_labels，然后按空格切词取词首字符标签。
    """
    samples = []
    for entry in entries:
        raw   = entry["raw"]
        spans = entry.get("spans", [])

        char_labels = ["O"] * len(raw)
        for span in spans:
            text  = span["text"]
            label = span["label"]
            idx   = raw.find(text)
            if idx == -1:
                continue
            char_labels[idx] = f"B-{label}"
            for i in range(idx + 1, idx + len(text)):
                char_labels[i] = f"I-{label}"

        tokens = []
        labels = []
        pos = 0
        for word in raw.split():
            idx = raw.find(word, pos)
            if idx == -1:
                tokens.append(word); labels.append("O")
                pos += len(word); continue
            tokens.append(word)
            labels.append(char_labels[idx])
            pos = idx + len(word)

        samples.append((tokens, labels))
    return samples


# ── 2. Construire le vocabulaire ──────────────────────────────────────────────
def build_vocab(train_samples):
    """
    从训练集构建词表。
    特殊符号：<PAD>（填充）, <UNK>（未知词/OOV）
    """
    word2id = {"<PAD>": 0, "<UNK>": 1}
    for tokens, _ in train_samples:
        for w in tokens:
            if w not in word2id:
                word2id[w] = len(word2id)
    return word2id


# ── 3. Dataset ────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    """
    将词序列转为 id 序列。
    未知词（测试集中未在训练集出现的词）映射到 <UNK>。
    """
    def __init__(self, samples, word2id):
        self.data    = samples
        self.word2id = word2id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]
        token_ids = [self.word2id.get(w, self.word2id["<UNK>"]) for w in tokens]
        label_ids = [LABEL2ID[l] for l in labels]
        return torch.tensor(token_ids, dtype=torch.long), \
               torch.tensor(label_ids, dtype=torch.long)


def collate_fn(batch):
    """
    动态 padding：将一个 batch 内的序列 pad 到该 batch 最大长度。
    比固定 MAX_LEN 更高效。
    """
    token_seqs, label_seqs = zip(*batch)
    token_padded = pad_sequence(token_seqs, batch_first=True, padding_value=0)
    label_padded = pad_sequence(label_seqs, batch_first=True, padding_value=-1)
    mask = (token_padded != 0)   # True = 真实 token，False = PAD
    return token_padded, label_padded, mask


# ── 4. Modèle BiLSTM-CRF ────────────────────────────────────────────────────
class BiLSTM_CRF(nn.Module):
    """
    BiLSTM-CRF 序列标注模型。

    架构：
      Embedding → Dropout → BiLSTM → Dropout → Linear → CRF

    CRF 层维护一个 (NUM_LABELS × NUM_LABELS) 的转移矩阵，
    学习标签之间的合法转移（如禁止 I-auteur 跟在 B-date 后面）。
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, dropout):
        super().__init__()
        self.num_labels = num_labels

        # 词嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout   = nn.Dropout(dropout)

        # 双向 LSTM：输出维度 = 2 × hidden_dim
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                            bidirectional=True, num_layers=1)

        # 将 BiLSTM 输出映射到标签空间（发射分数）
        self.hidden2tag = nn.Linear(hidden_dim * 2, num_labels)

        # CRF 转移矩阵：transitions[i][j] = 从标签 j 转移到标签 i 的分数
        self.transitions = nn.Parameter(torch.randn(num_labels, num_labels))
        # 初始化：禁止转移到 O 标签的中间位置（软约束）
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def get_emissions(self, x):
        """计算发射分数（BiLSTM 输出）"""
        emb  = self.dropout(self.embedding(x))
        out, _ = self.lstm(emb)
        out  = self.dropout(out)
        return self.hidden2tag(out)   # (batch, seq_len, num_labels)

    def crf_forward(self, emissions, mask):
        """
        CRF 前向算法：计算所有可能路径的归一化因子 log Z。
        用于训练时计算 negative log-likelihood。
        """
        batch_size, seq_len, _ = emissions.shape
        # 初始化：第一个时间步的分数
        score = emissions[:, 0, :]   # (batch, num_labels)
        for t in range(1, seq_len):
            # broadcast：当前分数 + 转移分数 + 发射分数
            score_t = score.unsqueeze(2) + self.transitions.unsqueeze(0) \
                      + emissions[:, t, :].unsqueeze(1)
            score_t = torch.logsumexp(score_t, dim=1)
            # mask：PAD 位置不更新分数
            mask_t  = mask[:, t].unsqueeze(1).float()
            score   = score_t * mask_t + score * (1 - mask_t)
        return torch.logsumexp(score, dim=1)   # (batch,)

    def score_sequence(self, emissions, tags, mask):
        """
        计算给定标签序列的路径分数（发射分数 + 转移分数之和）。
        """
        batch_size, seq_len, _ = emissions.shape
        score = torch.zeros(batch_size, device=emissions.device)
        for t in range(seq_len):
            mask_t = mask[:, t].float()
            emit   = emissions[:, t, :].gather(1, tags[:, t].unsqueeze(1)).squeeze(1)
            if t > 0:
                trans = self.transitions[tags[:, t], tags[:, t-1]]
                score = score + (emit + trans) * mask_t
            else:
                score = score + emit * mask_t
        return score

    def neg_log_likelihood(self, x, tags, mask):
        """
        训练损失：负对数似然 = log Z - score(正确路径)
        """
        emissions = self.get_emissions(x)
        # 将 PAD 位置的标签（-1）替换为 0，避免索引越界
        tags_safe = tags.clone()
        tags_safe[tags_safe < 0] = 0
        forward   = self.crf_forward(emissions, mask)
        gold      = self.score_sequence(emissions, tags_safe, mask)
        return (forward - gold).mean()

    def viterbi_decode(self, x, mask):
        """
        Viterbi 算法：找到最优标签序列（推理阶段）。
        """
        emissions  = self.get_emissions(x)
        batch_size, seq_len, _ = emissions.shape

        viterbi  = emissions[:, 0, :]
        backptr  = []

        for t in range(1, seq_len):
            v_t   = viterbi.unsqueeze(2) + self.transitions.unsqueeze(0)
            best_scores, best_tags = v_t.max(dim=1)
            backptr.append(best_tags)
            mask_t  = mask[:, t].unsqueeze(1).float()
            viterbi = (best_scores + emissions[:, t, :]) * mask_t \
                      + viterbi * (1 - mask_t)

        # 回溯
        best_last = viterbi.argmax(dim=1)
        best_path = [best_last]
        for bp in reversed(backptr):
            best_last = bp.gather(1, best_last.unsqueeze(1)).squeeze(1)
            best_path.append(best_last)
        best_path.reverse()
        best_path = torch.stack(best_path, dim=1)   # (batch, seq_len)
        return best_path


# ── 5. Évaluation ─────────────────────────────────────────────────────────────
def bio_to_spans(tokens, bio):
    spans = []
    cur_text, cur_label = [], None
    for tok, tag in zip(tokens, bio):
        if tag.startswith("B-"):
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text  = [tok]
            cur_label = tag[2:]
        elif tag.startswith("I-") and cur_text:
            cur_text.append(tok)
        else:
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text, cur_label = [], None
    if cur_text:
        spans.append({"text": " ".join(cur_text), "label": cur_label})
    return spans


def evaluate(model, loader, split_name="val", entries=None):
    """seqeval 实体级别评估 + détails par entrée optionnels."""
    model.eval()
    all_preds = []
    all_trues = []
    with torch.no_grad():
        for token_ids, label_ids, mask in loader:
            token_ids = token_ids.to(device)
            mask      = mask.to(device)
            preds     = model.viterbi_decode(token_ids, mask)

            for pred_seq, true_seq, m in zip(
                    preds.cpu().tolist(), label_ids.tolist(), mask.cpu().tolist()):
                p_tags = [ID2LABEL[p] for p, valid in zip(pred_seq, m) if valid]
                t_tags = [ID2LABEL[t] for t, valid in zip(true_seq, m) if valid]
                all_preds.append(p_tags)
                all_trues.append(t_tags)

    f1  = f1_score(all_trues, all_preds, zero_division=0)
    pre = precision_score(all_trues, all_preds, zero_division=0)
    rec = recall_score(all_trues, all_preds, zero_division=0)
    report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
    print(f"  [{split_name}] F1={f1:.4f}  P={pre:.4f}  R={rec:.4f}")
    print(classification_report(all_trues, all_preds, zero_division=0))

    details = []
    if entries is not None:
        for entry, gold_bio, pred_bio in zip(entries, all_trues, all_preds):
            tokens = entry["raw"].split()
            details.append({
                "entry_id":   entry["entry_id"],
                "raw":        entry["raw"],
                "gold_spans": bio_to_spans(tokens, gold_bio),
                "pred_spans": bio_to_spans(tokens, pred_bio),
            })

    return {"f1": f1, "precision": pre, "recall": rec,
            "report": report, "details": details}


# ── 6. Chargement des données ─────────────────────────────────────────────────
print("\n[1/5] Chargement des données...")
splits = {}
for split in ["train", "val", "test"]:
    with open(os.path.join(DATA_DIR, f"{split}.json"), encoding="utf-8") as f:
        splits[split] = json.load(f)
    print(f"  {split}: {len(splits[split])} entrées")

train_samples = json_to_bio(splits["train"])
val_samples   = json_to_bio(splits["val"])
test_samples  = json_to_bio(splits["test"])

# ── 7. Vocabulaire ────────────────────────────────────────────────────────────
print("\n[2/5] Construction du vocabulaire...")
word2id = build_vocab(train_samples)
print(f"  Vocab size: {len(word2id)} mots")

train_dataset = NERDataset(train_samples, word2id)
val_dataset   = NERDataset(val_samples,   word2id)
test_dataset  = NERDataset(test_samples,  word2id)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, collate_fn=collate_fn)

# ── 8. Modèle ─────────────────────────────────────────────────────────────────
print("\n[3/5] Initialisation du modèle BiLSTM-CRF...")
model = BiLSTM_CRF(
    vocab_size  = len(word2id),
    embed_dim   = EMBED_DIM,
    hidden_dim  = HIDDEN_DIM,
    num_labels  = NUM_LABELS,
    dropout     = DROPOUT,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Paramètres entraînables: {total_params:,}")

# ── 9. Entraînement ───────────────────────────────────────────────────────────
print(f"\n[4/5] Entraînement ({EPOCHS} epochs)...")
log_path = os.path.join(LOG_DIR, "training_log.jsonl")
best_f1  = 0.0
training_start = time.time()

with open(log_path, "w", encoding="utf-8") as logf:
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0

        for token_ids, label_ids, mask in train_loader:
            token_ids = token_ids.to(device)
            label_ids = label_ids.to(device)
            mask      = mask.to(device)

            # CRF 的 loss：负对数似然
            loss = model.neg_log_likelihood(token_ids, label_ids, mask)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        avg_loss   = total_loss / len(train_loader)
        epoch_time = time.time() - epoch_start
        timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        print(f"\nEpoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  "
              f"time={epoch_time:.1f}s  [{timestamp}]")
        val_metrics = evaluate(model, val_loader, "val")

        # 保存最佳模型
        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            torch.save(model.state_dict(),
                       os.path.join(MODEL_DIR, "model.pt"))
            # 保存词表（推理时需要）
            with open(os.path.join(MODEL_DIR, "word2id.json"), "w") as f:
                json.dump(word2id, f, ensure_ascii=False)
            print(f"  ✓ Meilleur modèle sauvegardé (F1={best_f1:.4f})")

        log_entry = {
            "epoch":         epoch,
            "timestamp":     timestamp,
            "duration_s":    round(epoch_time, 2),
            "train_loss":    round(avg_loss, 6),
            "val_f1":        round(val_metrics["f1"], 6),
            "val_precision": round(val_metrics["precision"], 6),
            "val_recall":    round(val_metrics["recall"], 6),
        }
        logf.write(json.dumps(log_entry, ensure_ascii=False) + "\n")
        logf.flush()

total_time = time.time() - training_start
print(f"\nEntraînement terminé en {total_time/60:.1f} min | "
      f"Meilleur val F1 = {best_f1:.4f}")

# ── 10. Évaluation finale sur test ───────────────────────────────────────────
print("\n[5/5] Évaluation sur le test set...")
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "model.pt"),
                                  map_location=device))
test_metrics = evaluate(model, test_loader, "test", entries=splits["test"])
val_metrics_final = evaluate(model, val_loader, "val", entries=splits["val"])

# ── Rapport Markdown ──────────────────────────────────────────────────────────
LABELS_ORDER = ["auteur", "ouvrage", "material", "date", "etat"]

def error_type(gold_spans, pred_spans):
    gold_set = {(s["text"], s["label"]) for s in gold_spans}
    pred_set = {(s["text"], s["label"]) for s in pred_spans}
    if not gold_set and not pred_set: return "correct_vide"
    if gold_set == pred_set: return "correct_exact"
    if gold_set and not pred_set: return "faux_negatif"
    if pred_set and not gold_set: return "faux_positif"
    return "erreur_partielle"

def make_markdown_report(split_name, metrics, details, report, out_dir=None):
    lines = []
    lines.append(f"# Rapport d'analyse — BiLSTM-CRF ({split_name.upper()})\n")
    lines.append(f"**Paramètres** : epochs={EPOCHS}, lr={LR}, batch={BATCH_SIZE}, "
                 f"embed={EMBED_DIM}, hidden={HIDDEN_DIM}  ")
    lines.append(f"**Référence** : Torres Aguilar (2022), LREC LT4HALA  ")
    lines.append(f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    lines.append("---\n")
    lines.append("## Matrice de résultats (seqeval — entité stricte)\n")
    lines.append("```")
    lines.append(f"{'Label':<15} {'P':>8} {'R':>8} {'F1':>8} {'Support':>9}")
    lines.append("-" * 52)
    for lab in LABELS_ORDER:
        r = report.get(lab, {})
        lines.append(f"{lab:<15} {r.get('precision',0):>8.3f} {r.get('recall',0):>8.3f} "
                     f"{r.get('f1-score',0):>8.3f} {int(r.get('support',0)):>9}")
    lines.append("-" * 52)
    for avg in ["micro avg", "macro avg", "weighted avg"]:
        r = report.get(avg, {})
        lines.append(f"{avg:<15} {r.get('precision',0):>8.3f} {r.get('recall',0):>8.3f} "
                     f"{r.get('f1-score',0):>8.3f} {int(r.get('support',0)):>9}")
    lines.append("```\n")
    lines.append(f"- **F1** = {metrics['f1']:.4f}  **P** = {metrics['precision']:.4f}  **R** = {metrics['recall']:.4f}\n")
    lines.append("---\n")
    lines.append("## Exemples détaillés par type d'erreur\n")
    grouped = {"correct_exact":[], "correct_vide":[], "faux_negatif":[],
               "faux_positif":[], "erreur_partielle":[]}
    for d in details:
        grouped[error_type(d["gold_spans"], d["pred_spans"])].append(d)
    labels_fr = {
        "correct_exact":   "✅ Correct — correspondance exacte",
        "correct_vide":    "✅ Correct — aucune entité attendue ni prédite",
        "faux_negatif":    "❌ Faux négatif — entité non détectée",
        "faux_positif":    "⚠️ Faux positif — entité prédite sans gold",
        "erreur_partielle":"〰️ Erreur partielle — détection incomplète ou mauvais label",
    }
    for key, title in labels_fr.items():
        ents = grouped[key]
        if not ents: continue
        lines.append(f"### {title} ({len(ents)} entrées)\n")
        for d in ents:
            gold_str = ", ".join(f"{s['label']}({s['text']})" for s in d["gold_spans"]) or "—"
            pred_str = ", ".join(f"{s['label']}({s['text']})" for s in d["pred_spans"]) or "—"
            lines.append(f"**#{d['entry_id']}** `{d['raw'][:90]}`")
            lines.append(f"- Gold : {gold_str}")
            lines.append(f"- Préd : {pred_str}\n")
    report_content = "\n".join(lines) + "\n"
    if out_dir:
        md_path = os.path.join(out_dir, f"analysis_{split_name}.md")
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        print(f"  → Rapport {split_name} : {md_path}")
        json_path = os.path.join(out_dir, f"results_{split_name}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({"split": split_name, "metrics": metrics,
                       "report_by_label": report, "details": details},
                      f, ensure_ascii=False, indent=2, cls=NumpyEncoder)
        print(f"  → JSON {split_name}    : {json_path}")
    return report_content


make_markdown_report("val",  val_metrics_final, val_metrics_final["details"],
                     val_metrics_final["report"], VAL_DIR)
make_markdown_report("test", test_metrics, test_metrics["details"],
                     test_metrics["report"], TEST_DIR)

# Rapport global (val + test ensemble)
md_lines = []
md_lines.append("# Rapport d'analyse — BiLSTM-CRF (paramètres article)\n")
md_lines.append(f"**Paramètres** : epochs={EPOCHS}, lr={LR}, batch={BATCH_SIZE}  ")
md_lines.append(f"**Référence** : Torres Aguilar (2022), LREC LT4HALA  ")
md_lines.append(f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
md_lines.append("---\n")
md_lines.append(make_markdown_report("val",  val_metrics_final, val_metrics_final["details"], val_metrics_final["report"]))
md_lines.append("---\n")
md_lines.append(make_markdown_report("test", test_metrics, test_metrics["details"], test_metrics["report"]))

md_path = os.path.join(LOG_DIR, "analysis_article_params.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines) + "\n")

# Save JSON results
results = {
    "model":      "BiLSTM-CRF",
    "note":       "Paramètres article Torres Aguilar 2022 : epochs=5, lr=0.01, batch=16",
    "epochs":     EPOCHS,
    "lr":         LR,
    "embed_dim":  EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "batch_size": BATCH_SIZE,
    "seed":       SEED,
    "vocab_size": len(word2id),
    "best_val_f1": round(best_f1, 6),
    "val_metrics": {
        "f1":        round(val_metrics_final["f1"], 6),
        "precision": round(val_metrics_final["precision"], 6),
        "recall":    round(val_metrics_final["recall"], 6),
    },
    "val_report_by_label":  val_metrics_final["report"],
    "val_details":          val_metrics_final["details"],
    "test_metrics": {
        "f1":        round(test_metrics["f1"], 6),
        "precision": round(test_metrics["precision"], 6),
        "recall":    round(test_metrics["recall"], 6),
    },
    "test_report_by_label": test_metrics["report"],
    "test_details":         test_metrics["details"],
    "total_training_time_min": round(total_time / 60, 2),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

with open(os.path.join(LOG_DIR, "test_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)

print(f"\n[Done] Val  F1={val_metrics_final['f1']:.4f}")
print(f"       Test F1={test_metrics['f1']:.4f}")
print(f"       Rapport global → {md_path}")
print(f"       Val            → {VAL_DIR}/")
print(f"       Test           → {TEST_DIR}/")
print(f"       Modèle         → {MODEL_DIR}/model.pt")


[Config] Device    : cuda
[Config] Epochs    : 5, LR: 0.01, Batch: 16
[Config] Embed_dim : 64, Hidden_dim: 128

[1/5] Chargement des données...
  train: 70 entrées
  val: 20 entrées
  test: 10 entrées

[2/5] Construction du vocabulaire...
  Vocab size: 509 mots

[3/5] Initialisation du modèle BiLSTM-CRF...
  Paramètres entraînables: 234,180

[4/5] Entraînement (5 epochs)...

Epoch 1/5  loss=28.5366  time=0.2s  [2026-06-07 22:13:35]
  [val] F1=0.4173  P=0.5577  R=0.3333
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00        17
        date       0.73      0.80      0.76        20
        etat       0.00      0.00      0.00         6
    material       0.48      0.62      0.54        21
     ouvrage       0.00      0.00      0.00        23

   micro avg       0.56      0.33      0.42        87
   macro avg       0.24      0.28      0.26        87
weighted avg       0.28      0.33      0.31        87

  ✓ Meilleur modèle sauvegardé (F1=0.